In [ ]:
!kaggle datasets download -d hsankesara/flickr-image-dataset

Dataset URL: https://www.kaggle.com/datasets/hsankesara/flickr-image-dataset
License(s): CC0-1.0
100% 8.16G/8.16G [01:58<00:00, 74.2MB/s]



In [ ]:
import zipfile
import os

with zipfile.ZipFile('flickr-image-dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('flickr30k_images')

print(os.listdir('flickr30k_images'))

['flickr30k_images']


In [ ]:
import os
import subprocess

REPO_DIR = "flickr30k_entities-master"

if not os.path.exists(REPO_DIR):
    subprocess.run(["wget", "-O", "master.zip", "https://github.com/BryanPlummer/flickr30k_entities/archive/refs/heads/master.zip"], check=True)
    subprocess.run(["unzip", "-o", "master.zip"], check=True)
    os.remove("master.zip")

In [ ]:
"""
Deep-Spec Exhaustive Ablation Study Benchmark — BLIP (Flickr30k Entities)
==================================================================================
this script isolates the Deep-Spec methodology and systematically ablates every
mathematical design choice: Laplacian types, Low-Rank K values, Eigengap heuristic,
Layer aggregation, and Non-linear sharpening.
"""

import torch
import numpy as np
from scipy.linalg import eigh
from transformers import BlipProcessor, BlipForImageTextRetrieval
from tqdm import tqdm
import warnings
import logging
import os
import zipfile
import xml.etree.ElementTree as ET
from collections import defaultdict
from PIL import Image
import re
import random
import subprocess


IMAGE_DIR = "/content/flickr30k_images/flickr30k_images/flickr30k_images"
REPO_DIR = "flickr30k_entities-master"
NUM_EVALS = 1000
SHUFFLE = True

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading BLIP Foundation Model...")
MODEL_NAME = "Salesforce/blip-itm-large-coco"
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(device)
model.eval()
GRID_SIZE = 24

def parse_phrases(sent_path):
    """Extract chain_id -> phrase mapping from Sentences file."""
    chain_to_phrase = {}
    if not os.path.exists(sent_path):
        return chain_to_phrase
    with open(sent_path, 'r') as f:
        lines = f.readlines()
        for line in lines[1:]:
            pattern = r'\[/EN#(\d+)/[^ ]+ (.*?)\]'
            matches = re.findall(pattern, line)
            for chain_id, phrase in matches:
                chain_to_phrase[chain_id] = phrase.strip()
    return chain_to_phrase

if not os.path.exists(REPO_DIR):
    print("Downloading official annotation repository...")
    subprocess.run(["wget", "-O", "master.zip", "https://github.com/BryanPlummer/flickr30k_entities/archive/refs/heads/master.zip"], check=True)
    subprocess.run(["unzip", "-o", "master.zip"], check=True)
    os.remove("master.zip")

ANNO_ZIP = os.path.join(REPO_DIR, "annotations.zip")
if os.path.exists(ANNO_ZIP) and not os.path.exists(os.path.join(REPO_DIR, "Annotations")):
    print("Extracting annotations.zip...")
    with zipfile.ZipFile(ANNO_ZIP, 'r') as zip_ref:
        zip_ref.extractall(REPO_DIR)

with open(os.path.join(REPO_DIR, "test.txt"), "r") as f:
    test_image_ids = [line.strip().split('.')[0] for line in f]

image_to_objects = defaultdict(list)
for img_id in test_image_ids:
    xml_path = os.path.join(REPO_DIR, "Annotations", f"{img_id}.xml")
    if not os.path.exists(xml_path):
        continue
    try:
        tree = ET.parse(xml_path)
    except:
        continue
    root = tree.getroot()

    sent_path = os.path.join(REPO_DIR, "Sentences", f"{img_id}.txt")
    chain_to_phrase = parse_phrases(sent_path)

    for obj in root.iter('object'):
        chain_id = obj.find('name').text
        bbox_elem = obj.find('bndbox')
        if bbox_elem is None:
            continue
        xmin = int(bbox_elem.find('xmin').text)
        ymin = int(bbox_elem.find('ymin').text)
        xmax = int(bbox_elem.find('xmax').text)
        ymax = int(bbox_elem.find('ymax').text)
        phrase = chain_to_phrase.get(chain_id, chain_id)
        image_to_objects[img_id].append({
            'category_name': phrase,
            'bbox': [xmin, ymin, xmax - xmin, ymax - ymin]
        })

print(f"Loaded annotations for {len(image_to_objects)} test images.")

ablation_methods = [
    "Baseline (No Denoising, Raw Avg)",
    "DS: Default (L_sym, K=5, Global Mean, Abs)",
    "DS: Eigengap Heuristic (Autotune K)",
    "DS: Low-Rank K=2 (Under-param)",
    "DS: K=Full (No Spectral Truncation)",
    "DS: Unnormalized Laplacian (D-W)",
    "DS: Random Walk Laplacian",
    "DS: Last Layer Only (No Gestalt)",
    "DS: Markov Sharpening (C squared)",
    "DS: ReLU instead of Abs (Sign fix)"
]
hits = {m: 0 for m in ablation_methods}
total_evaluated = 0


def extract_ablation_heatmaps(inputs, valid_indices, tgt_rel):
    """
    Extracts all ablated versions of the Deep-Spec methodology in a single pass.
    """
    heatmaps = {}

    attn_maps = []
    hooks = []
    def hook_fn(module, inp, out):
        attn_maps.append(inp[0].detach().cpu())

    for name, module in model.named_modules():
        if "crossattention.self.dropout" in name:
            hooks.append(module.register_forward_hook(hook_fn))

    with torch.no_grad():
        _ = model(**inputs)

    for h in hooks:
        h.remove()

    all_layers_attn = torch.stack(attn_maps)

    # Core Tensors for ablations
    A_global = all_layers_attn.mean(dim=(0, 1, 2))[valid_indices, 1:]
    A_global_norm = A_global / (A_global.max() + 1e-10)

    A_last = all_layers_attn[-1].mean(dim=(0, 1))[valid_indices, 1:]
    A_last_norm = A_last / (A_last.max() + 1e-10)

    # ABLATION 1: Baseline (Raw Global Average, No Graph Theory)
    hm_raw = A_global_norm[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
    heatmaps["Baseline (No Denoising, Raw Avg)"] = hm_raw.numpy()

    # Generic Spectral Processor to test all math variations
    def compute_spectral_variant(base_W, laplacian="sym", k_val=5, use_abs=True, apply_sharpening=False):
        W_c = base_W.numpy().copy()

        # Ablation: Markov Path Sharpening (C squared)
        if apply_sharpening:
            W_c = W_c ** 2

        T_ds, I_ds = W_c.shape
        N = T_ds + I_ds
        W = np.zeros((N, N))
        W[:T_ds, T_ds:] = W_c
        W[T_ds:, :T_ds] = W_c.T
        W = W + 1e-3  # Connectivity regularizer

        D = np.sum(W, axis=1)

        # Ablation: Laplacian Formulation
        if laplacian == "sym":
            D_inv_sqrt = np.diag(1.0 / np.sqrt(D))
            L = np.eye(N) - D_inv_sqrt @ W @ D_inv_sqrt
        elif laplacian == "unnorm":
            L = np.diag(D) - W
        elif laplacian == "rw":
            D_inv = np.diag(1.0 / np.maximum(D, 1e-10))
            L = np.eye(N) - D_inv @ W

        evals, eigvecs = eigh(L)

        # Ablation: Truncation (K values)
        if k_val == "full":
            k = N - 2
        elif k_val == "eigengap":
            gaps = np.diff(evals[1:16])
            k = np.argmax(gaps) + 1
        else:
            k = k_val

        V_text = eigvecs[:T_ds, 1:k+1]
        V_img = eigvecs[T_ds:, 1:k+1]
        denoised_affinity = V_text @ V_img.T

        if use_abs:
            target_affinity = np.abs(denoised_affinity[tgt_rel, :])
        else:
            target_affinity = np.maximum(0, denoised_affinity[tgt_rel, :]) # ReLU equivalent

        hm = target_affinity.max(axis=0).reshape(GRID_SIZE, GRID_SIZE)
        if hm.max() - hm.min() == 0:
            return hm
        return (hm - hm.min()) / (hm.max() - hm.min() + 1e-10)

    heatmaps["DS: Default (L_sym, K=5, Global Mean, Abs)"] = compute_spectral_variant(A_global_norm)
    heatmaps["DS: Eigengap Heuristic (Autotune K)"]        = compute_spectral_variant(A_global_norm, k_val="eigengap")
    heatmaps["DS: Low-Rank K=2 (Under-param)"]             = compute_spectral_variant(A_global_norm, k_val=2)
    heatmaps["DS: K=Full (No Spectral Truncation)"]        = compute_spectral_variant(A_global_norm, k_val="full")
    heatmaps["DS: Unnormalized Laplacian (D-W)"]           = compute_spectral_variant(A_global_norm, laplacian="unnorm")
    heatmaps["DS: Random Walk Laplacian"]                  = compute_spectral_variant(A_global_norm, laplacian="rw")
    heatmaps["DS: Last Layer Only (No Gestalt)"]           = compute_spectral_variant(A_last_norm)
    heatmaps["DS: Markov Sharpening (C squared)"]          = compute_spectral_variant(A_global_norm, apply_sharpening=True)
    heatmaps["DS: ReLU instead of Abs (Sign fix)"]         = compute_spectral_variant(A_global_norm, use_abs=False)

    return heatmaps


print(f"Starting Deep-Spec Ablation Benchmark over {NUM_EVALS} Flickr30k Entities...\n")

if SHUFFLE:
    random.shuffle(test_image_ids)

evaluated = 0
with tqdm(total=NUM_EVALS, desc="Evaluating", unit="obj") as pbar:
    for img_id in test_image_ids:
        if evaluated >= NUM_EVALS:
            break
        if img_id not in image_to_objects:
            continue
        objects = image_to_objects[img_id]

        img_path = None
        for ext in ['.jpg', '.jpeg', '.png']:
            candidate = os.path.join(IMAGE_DIR, f"{img_id}{ext}")
            if os.path.exists(candidate):
                img_path = candidate
                break
        if img_path is None:
            continue

        try:
            image = Image.open(img_path).convert("RGB")
        except:
            continue
        orig_W, orig_H = image.size

        sent_path = os.path.join(REPO_DIR, "Sentences", f"{img_id}.txt")
        if not os.path.exists(sent_path):
            continue
        with open(sent_path, 'r') as f:
            caption = f.readline().strip()
        if not caption:
            continue

        target_noun = None
        for obj in objects:
            phrase = obj['category_name']
            if phrase.lower() in caption.lower():
                target_noun = phrase
                break
        if not target_noun:
            continue
        all_target_bboxes = [obj['bbox'] for obj in objects if obj['category_name'] == target_noun]

        inputs = processor(images=image, text=caption, return_tensors="pt").to(device)

        tokens = processor.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        special_ids = processor.tokenizer.all_special_ids
        valid_indices = [i for i, idx in enumerate(inputs["input_ids"][0].tolist()) if idx not in special_ids]
        target_indices = []
        for i, t in enumerate(tokens):
            if i in valid_indices:
                clean_token = t.lower().replace("##", "")
                if any(word in clean_token or clean_token in word for word in target_noun.lower().split()):
                    target_indices.append(i)
        if not target_indices:
            continue
        tgt_rel = [valid_indices.index(idx) for idx in target_indices]

        heatmaps = extract_ablation_heatmaps(inputs, valid_indices, tgt_rel)
        if heatmaps is None:
            continue

        for m_name in ablation_methods:
            hm = heatmaps[m_name]
            if hm.max() == 0:
                continue
            max_idx = np.argmax(hm)
            max_y_patch, max_x_patch = np.unravel_index(max_idx, hm.shape)
            rel_x = (max_x_patch + 0.5) / GRID_SIZE
            rel_y = (max_y_patch + 0.5) / GRID_SIZE
            pixel_x = rel_x * orig_W
            pixel_y = rel_y * orig_H

            hit = False
            for bbox in all_target_bboxes:
                x_min, y_min, bbox_w, bbox_h = bbox
                x_max, y_max = x_min + bbox_w, y_min + bbox_h
                if (x_min <= pixel_x <= x_max) and (y_min <= pixel_y <= y_max):
                    hit = True
                    break
            if hit:
                hits[m_name] += 1

        evaluated += 1
        pbar.update(1)

        if evaluated % 100 == 0:
            pbar.write(f"\n--- Ablation Progress ({evaluated}/{NUM_EVALS}) ---")
            for m_name in ablation_methods:
                acc = (hits[m_name] / evaluated) * 100
                pbar.write(f"{m_name:<45} | {acc:.1f}%")
            pbar.write("-" * 60)


print("\n" + "=" * 70)
print(" FINAL DEEP-SPEC ABLATION STUDY RESULTS ".center(70))
print("=" * 70)
print(f"{'Ablated Component / Variant':<45} | {'Pointing Acc (%)':<20}")
print("-" * 70)
for m_name in ablation_methods:
    acc = (hits[m_name] / evaluated) * 100 if evaluated > 0 else 0
    print(f"{m_name:<45} | {acc:.2f}%")
print("=" * 70)

Loading BLIP Foundation Model...


Loading weights:   0%|          | 0/616 [00:00<?, ?it/s]

Extracting annotations.zip...
Loaded annotations for 999 test images.
Starting Deep-Spec Ablation Benchmark over 1000 Flickr30k Entities...



Evaluating:  10%|█         | 100/1000 [04:00<27:37,  1.84s/obj]


--- Ablation Progress (100/1000) ---
Baseline (No Denoising, Raw Avg)              | 19.0%
DS: Default (L_sym, K=5, Global Mean, Abs)    | 57.0%
DS: Eigengap Heuristic (Autotune K)           | 10.0%
DS: Low-Rank K=2 (Under-param)                | 66.0%
DS: K=Full (No Spectral Truncation)           | 9.0%
DS: Unnormalized Laplacian (D-W)              | 12.0%
DS: Random Walk Laplacian                     | 59.0%
DS: Last Layer Only (No Gestalt)              | 54.0%
DS: Markov Sharpening (C squared)             | 56.0%
DS: ReLU instead of Abs (Sign fix)            | 61.0%
------------------------------------------------------------


...


--- Ablation Progress (700/1000) ---
Baseline (No Denoising, Raw Avg)              | 17.1%
DS: Default (L_sym, K=5, Global Mean, Abs)    | 60.9%
DS: Eigengap Heuristic (Autotune K)           | 14.6%
DS: Low-Rank K=2 (Under-param)                | 58.6%
DS: K=Full (No Spectral Truncation)           | 14.3%
DS: Unnormalized Laplacian (D-W)              | 14.4%
DS: Random Walk Laplacian                     | 61.0%
DS: Last Layer Only (No Gestalt)              | 50.3%
DS: Markov Sharpening (C squared)             | 53.7%
DS: ReLU instead of Abs (Sign fix)            | 58.6%
------------------------------------------------------------


Evaluating:  80%|████████  | 800/1000 [28:04<06:44,  2.02s/obj]


--- Ablation Progress (800/1000) ---
Baseline (No Denoising, Raw Avg)              | 16.5%
DS: Default (L_sym, K=5, Global Mean, Abs)    | 60.6%
DS: Eigengap Heuristic (Autotune K)           | 14.0%
DS: Low-Rank K=2 (Under-param)                | 57.6%
DS: K=Full (No Spectral Truncation)           | 14.1%
DS: Unnormalized Laplacian (D-W)              | 14.1%
DS: Random Walk Laplacian                     | 60.5%
DS: Last Layer Only (No Gestalt)              | 49.8%
DS: Markov Sharpening (C squared)             | 52.8%
DS: ReLU instead of Abs (Sign fix)            | 58.0%
------------------------------------------------------------


Evaluating:  87%|████████▋ | 872/1000 [30:29<04:28,  2.10s/obj]


                FINAL DEEP-SPEC ABLATION STUDY RESULTS                
Ablated Component / Variant                   | Pointing Acc (%)    
----------------------------------------------------------------------
Baseline (No Denoising, Raw Avg)              | 16.74%
DS: Default (L_sym, K=5, Global Mean, Abs)    | 60.09%
DS: Eigengap Heuristic (Autotune K)           | 14.33%
DS: Low-Rank K=2 (Under-param)                | 57.68%
DS: K=Full (No Spectral Truncation)           | 14.33%
DS: Unnormalized Laplacian (D-W)              | 13.99%
DS: Random Walk Laplacian                     | 59.98%
DS: Last Layer Only (No Gestalt)              | 50.11%
DS: Markov Sharpening (C squared)             | 52.41%
DS: ReLU instead of Abs (Sign fix)            | 57.45%
